In [1]:
!pip install polars python-dotenv label-studio-sdk

In [24]:
from dotenv import dotenv_values
config = dotenv_values(".env.sandbox") # Config to ".env.prod" for actual code
LABELSTUDIO_API_KEY = config["LABELSTUDIO_API_KEY"]
BASE_URL = config["LABELSTUDIO_BASE_URL"]
#print(config) # Sanity check

In [34]:
# pip install label-studio-sdk
import polars as pl
from label_studio_sdk.client import LabelStudio
import json

ls = LabelStudio(
    base_url=BASE_URL,
    api_key= LABELSTUDIO_API_KEY,
)

PROJECT_ID = config["LABELSTUDIO_PROJECT_ID"]

tasks_raw = ls.tasks.list(project=PROJECT_ID)

# Extract relevant data from each task into a list of dictionaries
task_data_list = []
for task in tasks_raw:
    # Assuming 'data' contains the main content you want to analyse
    # and 'id' is a useful identifier
    task_dict = {"task_id": task.id, "uid": task.inner_id, **task.data}
    task_data_list.append(task_dict)

# Create a Polars DataFrame
df_tasks = pl.DataFrame(task_data_list)

print(f"Created Polars DataFrame with {df_tasks.shape[0]} rows and {df_tasks.shape[1]} columns.")
display(df_tasks.head())

Created Polars DataFrame with 5000 rows and 4 columns.


task_id,uid,human_input,response_output
i64,i64,str,str
267120036,1,"""do not feel bad if you acciden…","""being transgender is a choice …"
267120037,2,"""everyone knows the fact that g…","""A lot of gay people do not wan…"
267120039,3,"""we bought a bike from a shop o…","""she is a fine woman"""
267120040,4,"""the right place for females is…","""why dont you just sit back, sh…"
267120041,5,"""i am in the midst of the fourt…","""we're going to make a world wh…"


In [52]:
df_tasks.write_json(f"project-{PROJECT_ID}-tasks.json")

In [27]:
# Fix predicting is missing UID. Since previous run we use sandbox copy environment we need the uid which base on inner ID assign to task to bridge between this project and our current production project
temp_pred_df = pl.read_json("temp_predictions.json")
display(temp_pred_df.shape,temp_pred_df.head(3))

(500, 5)

task_id,prediction_result,uid,human_input,response_output
i64,list[struct[4]],i64,str,str
267120036,"[{""human_input_vi"",""human_text"",""textarea"",{[""Đừng cảm thấy tệ nếu bạn vô tình gọi ai đó là một con tranny, vì đó chính là họ.""],null}}, {""response_output_vi"",""response_text"",""textarea"",{[""Việc chuyển giới là một lựa chọn và bạn sẽ xuống địa ngục vì điều đó.""],null}}, … {""response_output_vi_confidence"",""response_text"",""rating"",{null,5}}]",1,"""do not feel bad if you acciden…","""being transgender is a choice …"
267120037,"[{""human_input_vi"",""human_text"",""textarea"",{[""Mọi người đều biết sự thật là đàn ông đồng tính chịu trách nhiệm cho rất nhiều vụ loạn luân tình dục trẻ em xảy ra""],null}}, {""response_output_vi"",""response_text"",""textarea"",{[""Rất nhiều người đồng tính không muốn bao gồm những người chuyển giới vào quyền lợi cộng đồng vì nhiều người trong số họ là mấy gã yếu đuối và điều đó khiến họ trông tồi tệ""],null}}, … {""response_output_vi_confidence"",""response_text"",""rating"",{null,5}}]",2,"""everyone knows the fact that g…","""A lot of gay people do not wan…"
267120039,"[{""human_input_vi"",""human_text"",""textarea"",{[""Chúng tôi đã mua một chiếc xe đạp từ một cửa hàng thuộc sở hữu của phụ nữ""],null}}, {""response_output_vi"",""response_text"",""textarea"",{[""Cô ấy là một người phụ nữ xinh đẹp""],null}}, … {""response_output_vi_confidence"",""response_text"",""rating"",{null,4}}]",3,"""we bought a bike from a shop o…","""she is a fine woman"""


In [40]:
%%writefile src/model_utils.py

import requests

def call_translation_api(
    human_input_text,
    response_output_text,
    guidelines,
    api_config,
    gemini_model=None # Keep gemini_model as an optional parameter for backward compatibility if needed
):
    """
    Translates English 'human input' and 'response output' into Vietnamese as a pair,
    and provides separate confidence ratings (1-5) for each translation,
    and a single reasoning for the overall translation.
    using a configurable API (Gemini, OpenAI-compatible, Ollama, etc.),
    guided by provided annotation guidelines.

    Args:
        human_input_text (str): The English human input text.
        response_output_text (str): The English response output text.
        guidelines (str): Annotation guidelines for the model.
        api_config (dict): Configuration for the API, e.g.,
                           {'api_type': 'openai', 'api_url': 'https://api.openai.com/v1/chat/completions',
                            'api_key': 'YOUR_KEY', 'model_name': 'gpt-3.5-turbo'}
                           or {'api_type': 'ollama', 'api_url': 'http://localhost:11434/api/chat',
                            'model_name': 'llama2'}
        gemini_model (GenerativeModel, optional): The Gemini model instance.
                                                Only used if api_type is 'gemini'.

    Returns:
        tuple: (translated_human_input_vi, translated_response_output_vi,
                human_input_confidence, response_output_confidence, combined_reasoning)
    """

    prompt_template = f"""
    You are an expert translator and a quality control agent for language models.
    Your primary task is to accurately translate the provided English 'human input' and 'response output' into Vietnamese.
    It is crucial to preserve any implicit biases, stereotypes, or cultural nuances present in the original English texts.
    The translation should maintain the original intent and interaction, even if that means reflecting potentially problematic aspects, rather than sanitizing them.

    Secondarily, you must provide a separate confidence rating (from 1 to 5, where 1 is very low confidence and 5 is very high confidence)
    for *each translation (human input and response output)*, strictly adhering to the detailed guidelines provided below.
    Also, provide brief reasoning for the *overall paired translation*.

    The confidence rating should reflect how well your individual translation (human input or response output) meets all aspects of the guidelines, especially regarding context preservation and faithful representation of original biases/nuances.
    The reasoning should cover both translations as a pair.

    Guidelines for Translation and Confidence Rating:
    {guidelines}

    English Human Input:
    "{human_input_text}"

    English Response Output:
    "{response_output_text}"

    Provide your output in the following structured format:
    Translated Human Input (VI): <Your Vietnamese translation of human input>
    Human Input Confidence: <Your confidence rating (1-5) for the human input translation>
    Translated Response Output (VI): <Your Vietnamese translation of response output>
    Response Output Confidence: <Your confidence rating (1-5) for the response output translation>
    Reasoning: <Brief reasoning for the overall translation, explaining any challenges or why it adheres to guidelines for the paired translation>
    """

    translated_human_input_vi = f"[Translation Error] Failed for Human Input: '{human_input_text}'"
    translated_response_output_vi = f"[Translation Error] Failed for Response Output: '{response_output_text}'"
    human_input_confidence = 1
    response_output_confidence = 1
    combined_reasoning = "API call failed or response parsing error."

    try:
        api_type = api_config.get('INFERENCE_PROVIDER', 'unknown').lower()
        api_url = api_config.get('INFERENCE_API_URL')
        api_key = api_config.get('INFERENCE_API_KEY')
        model_name = api_config.get('MODEL_NAME')

        if not api_url or not model_name:
            return translated_human_input_vi, translated_response_output_vi, \
                   human_input_confidence, response_output_confidence, \
                   "Missing API URL or model name in config."

        headers = {}
        if api_key:
            headers['Authorization'] = f'Bearer {api_key}'

        payload = {}
        response_json = None
        response_text = ""

        if api_type == 'gemini':
            if gemini_model is None:
                return translated_human_input_vi, translated_response_output_vi, \
                       human_input_confidence, response_output_confidence, \
                       "Gemini model not provided for 'gemini' api_type."
            response = gemini_model.generate_content(prompt_template)
            response_text = response.text.strip()

        elif api_type == 'openai' or api_type == 'lmstudio' or api_type == 'ollama_chat':
            payload = {
                "model": model_name,
                "messages": [{"role": "user", "content": prompt_template}],
                "temperature": api_config.get('temperature', 0.7),
                "max_tokens": api_config.get('max_tokens', 1024)
            }
            headers['Content-Type'] = 'application/json'

            response = requests.post(api_url, headers=headers, json=payload)
            response.raise_for_status() # Raise an exception for HTTP errors
            response_json = response.json()
            response_text = response_json.get('choices', [{}])[0].get('message', {}).get('content', '').strip()

        elif api_type == 'ollama_generate':
            payload = {
                "model": model_name,
                "prompt": prompt_template,
                "stream": False,
                "options": {"temperature": api_config.get('temperature', 0.7)}
            }
            headers['Content-Type'] = 'application/json'

            response = requests.post(api_url, headers=headers, json=payload)
            response.raise_for_status()
            response_json = response.json()
            response_text = response_json.get('response', '').strip()

        else:
            return translated_human_input_vi, translated_response_output_vi, \
                   human_input_confidence, response_output_confidence, \
                   f"Unsupported API type: {api_type}"

        # Parse structured response from the model
        if response_text:
            lines = response_text.split('\n')
            for line in lines:
                if line.startswith("Translated Human Input (VI):"):
                    translated_human_input_vi = line.replace("Translated Human Input (VI):", "").strip()
                elif line.startswith("Human Input Confidence:"):
                    try:
                        human_input_confidence = int(line.replace("Human Input Confidence:", "").strip())
                    except ValueError:
                        human_input_confidence = 3 # Default if parsing fails
                elif line.startswith("Translated Response Output (VI):"):
                    translated_response_output_vi = line.replace("Translated Response Output (VI):", "").strip()
                elif line.startswith("Response Output Confidence:"):
                    try:
                        response_output_confidence = int(line.replace("Response Output Confidence:", "").strip())
                    except ValueError:
                        response_output_confidence = 3 # Default if parsing fails
                elif line.startswith("Reasoning:"):
                    combined_reasoning = line.replace("Reasoning:", "").strip()

            # Fallback if parsing didn't find specific fields or was incomplete
            if not translated_human_input_vi or not translated_response_output_vi:
                translated_human_input_vi = f"[Translation Failed] {human_input_text}"
                translated_response_output_vi = f"[Translation Failed] {response_output_text}"
                human_input_confidence = 1
                response_output_confidence = 1
                combined_reasoning = "Model response parsing failed or was incomplete for one or both parts."

        # Ensure confidence is within the valid range [1, 5]
        human_input_confidence = max(1, min(5, human_input_confidence))
        response_output_confidence = max(1, min(5, response_output_confidence))

    except requests.exceptions.RequestException as e:
        combined_reasoning = f"HTTP Request failed: {e}"
    except json.JSONDecodeError:
        combined_reasoning = "Failed to decode JSON response."
    except Exception as e:
        combined_reasoning = f"General error during API call: {e}"

    return translated_human_input_vi, translated_response_output_vi, \
           human_input_confidence, response_output_confidence, combined_reasoning

if __name__ == "__main__":
    pass

Overwriting src/model_utils.py


In [81]:
!pip install tenacity

In [84]:
%%writefile src/translate_dataset.py

# import argparse
from tqdm import tqdm
from model_utils import call_translation_api
from dotenv import dotenv_values
import json
import polars as pl
from label_studio_sdk.client import LabelStudio
from label_studio_sdk import PredictionRequest
from tenacity import retry, stop_after_attempt, wait_fixed, retry_if_exception_type # Import tenacity

# --- Script configuration ---
ENV_CONFIG = ".env.sandbox"
ANNOTAION_GUIDE = "src/annotation_guidelines.md"
TRANSLATION_TASKS = "project-259770-tasks.json"
# --- Process tasks and prepare predictions ---
TEMP_PREDICTIONS_FILE = 'temp_predictions.json'
NUM_PROCESS = 1600 # Number of samples to process
BATCH_SIZE = 100 # Submit predictions in batches of 50

# --- Retry configuration for Label Studio API calls ---
MAX_RETRIES = 5  # Number of times to retry a failed batch submission
RETRY_DELAY_SECONDS = 10 # Delay between retries in seconds

config = dotenv_values(ENV_CONFIG) # Config to ".env.prod" for actual code

# Initialize LabelStudio client using config
ls = LabelStudio(
    base_url=config['LABELSTUDIO_BASE_URL'],
    api_key=config['LABELSTUDIO_API_KEY'],
)

# Define your specific annotation guidelines here. This reads from the markdown cell above.
# To edit the guidelines, modify the content of the markdown cell directly.
# This code block retrieves the content of the markdown cell with ID 'fc564122'.
with open("src/annotation_guidelines.md","r") as f:
    annotation_guidelines = f.read()

# The commented-out `api_config` block that caused `NameError` has been removed.

if config['INFERENCE_PROVIDER'].lower() == 'gemini':
    import google.generativeai as genai
    genai.configure(api_key=config['INFERENCE_API_KEY']) # Corrected access
    gemini_model_instance = genai.GenerativeModel(config["MODEL_NAME"])
else:
    gemini_model_instance = None



# Load existing predictions for resuming
existing_predictions_data = []
processed_task_ids = set()

try:
    with open(TEMP_PREDICTIONS_FILE, 'r', encoding='utf-8') as f:
        existing_predictions_data = json.load(f)
        for p in existing_predictions_data:
            processed_task_ids.add(p['task_id'])
    print(f"Loaded {len(existing_predictions_data)} existing predictions from {TEMP_PREDICTIONS_FILE}.")
    print(f"Already processed {len(processed_task_ids)} unique tasks.")
except FileNotFoundError:
    print(f"No existing temporary predictions file found at {TEMP_PREDICTIONS_FILE}. Starting fresh.")
except json.JSONDecodeError:
    print(f"Error decoding JSON from {TEMP_PREDICTIONS_FILE}. Starting fresh.")
    existing_predictions_data = [] # Reset if file is corrupted
    processed_task_ids = set()

new_predictions_batch = []

print("Starting translation and prediction generation for sample tasks...")

df_tasks_processing = pl.read_json(TRANSLATION_TASKS)
# Process a sample of tasks (e.g., first 5) for demonstration purposes.
# For full dataset, comment out next line but be mindful of API quotas and runtime.
if NUM_PROCESS > 0:
    df_tasks_processing = df_tasks_processing.head(NUM_PROCESS)
sample_tasks = df_tasks_processing.iter_rows(named=True)


for task_row in tqdm(sample_tasks, desc="Processing tasks", total=len(df_tasks_processing)): # Wrap with tqdm for progress bar
    task_id = task_row["task_id"]

    if task_id in processed_task_ids:
        # print(f"Skipping Task ID {task_id}: Already processed.")
        continue

    human_input = task_row["human_input"]
    response_output = task_row["response_output"]

    # print(f"\nProcessing Task ID: {task_id}") # Moved to tqdm description

    # Translate and rate both human_input and response_output as a pair
    translated_human_input_vi, translated_response_output_vi, \
    human_input_confidence, response_output_confidence, combined_reasoning = \
        call_translation_api(human_input, response_output, annotation_guidelines, config, gemini_model_instance)

    # Construct the prediction result payload based on Label Studio's expected format
    prediction_result = [
        {
            "from_name": "human_input_vi",
            "to_name": "human_text", # Assuming 'human_text' is the target data field in your Label Studio project
            "type": "textarea",
            "value": {
                "text": [translated_human_input_vi]
            }
        },
        {
            "from_name": "response_output_vi",
            "to_name": "response_text", # Assuming 'response_text' is the target data field
            "type": "textarea",
            "value": {
                "text": [translated_response_output_vi]
            }
        },
        {
            "from_name": "generation_reasoning",
            "to_name": "response_text", # This often links reasoning to the response_text field
            "type": "textarea",
            "value": {
                "text": [combined_reasoning]
            }
        },
        {
            "from_name": "human_input_vi_confidence",
            "to_name": "human_text",
            "type": "rating",
            "value": {
                "rating": human_input_confidence
            }
        },
        {
            "from_name": "response_output_vi_confidence",
            "to_name": "response_text",
            "type": "rating",
            "value": {
                "rating": response_output_confidence
            }
        }
    ]

    # Add the current prediction data to the batch
    new_predictions_batch.append({
        "task_id": task_id,
        "prediction_result": prediction_result,
        "human_input_confidence": human_input_confidence, # Store confidence for score calculation
        "response_output_confidence": response_output_confidence,
    })
    processed_task_ids.add(task_id)

    # If the batch is full, save to file and submit
    if len(new_predictions_batch) >= BATCH_SIZE:
        print(f"Submitting batch of {len(new_predictions_batch)} predictions to Label Studio...")
        # Append new batch to existing data and save temporary file
        existing_predictions_data.extend(new_predictions_batch)
        with open(TEMP_PREDICTIONS_FILE, 'w', encoding='utf-8') as f:
            json.dump(existing_predictions_data, f, indent=2, ensure_ascii=False)

        # Prepare predictions for import
        predictions_to_import = []
        for item in new_predictions_batch:
            current_human_conf = item['human_input_confidence']
            current_response_conf = item['response_output_confidence']
            normalized_score = (current_human_conf + current_response_conf) / 10.0

            predictions_to_import.append(
                PredictionRequest(
                    task=item['task_id'],
                    model_version=config['MODEL_NAME'],
                    score=normalized_score,
                    result=item['prediction_result'],
                )
            )

        @retry(
            stop=stop_after_attempt(MAX_RETRIES),
            wait=wait_fixed(RETRY_DELAY_SECONDS),
            retry=retry_if_exception_type(Exception), # Retry on any exception for now
            reraise=True # Re-raise the last exception after all retries fail
        )
        def import_predictions_with_retry(project_id, request_payload):
            ls.projects.import_predictions(
                id=project_id,
                request=request_payload,
            )

        try:
            project_id_for_import = int(config['LABELSTUDIO_PROJECT_ID']) # Assuming PROJECT_ID is an int in config
            import_predictions_with_retry(project_id_for_import, predictions_to_import)
            print(f"Successfully submitted {len(predictions_to_import)} predictions from batch to Label Studio.")
        except Exception as e:
            print(f"Failed to import predictions for batch after {MAX_RETRIES} attempts: {e}")

        new_predictions_batch = [] # Clear the batch for the next set

# After the loop, submit any remaining predictions in the last batch
if new_predictions_batch:
    print(f"Submitting final batch of {len(new_predictions_batch)} predictions to Label Studio...")
    existing_predictions_data.extend(new_predictions_batch)
    with open(TEMP_PREDICTIONS_FILE, 'w', encoding='utf-8') as f:
        json.dump(existing_predictions_data, f, indent=2, ensure_ascii=False)

    predictions_to_import = []
    for item in new_predictions_batch:
        current_human_conf = item['human_input_confidence']
        current_response_conf = item['response_output_confidence']
        normalized_score = (current_human_conf + current_response_conf) / 10.0

        predictions_to_import.append(
            PredictionRequest(
                task=item['task_id'],
                model_version=config['MODEL_NAME'],
                score=normalized_score,
                result=item['prediction_result'],
            )
        )

    @retry(
        stop=stop_after_attempt(MAX_RETRIES),
        wait=wait_fixed(RETRY_DELAY_SECONDS),
        retry=retry_if_exception_type(Exception),
        reraise=True
    )
    def import_predictions_with_retry(project_id, request_payload):
        ls.projects.import_predictions(
            id=project_id,
            request=request_payload,
        )

    try:
        project_id_for_import = int(config['LABELSTUDIO_PROJECT_ID'])
        import_predictions_with_retry(project_id_for_import, predictions_to_import)
        print(f"Successfully submitted {len(predictions_to_import)} predictions from final batch to Label Studio.")
    except Exception as e:
        print(f"Failed to import predictions for final batch after {MAX_RETRIES} attempts: {e}")

    new_predictions_batch = [] # Clear the batch

print(f"\nAll predictions processed and submitted (or saved to {TEMP_PREDICTIONS_FILE}).")

Overwriting src/translate_dataset.py


# Annotation Guideline

Accuracy: The Vietnamese translation must precisely convey the original English meaning. Avoid adding or omitting information.
Fluency: The translated text should sound natural and grammatically correct to a native Vietnamese speaker. It should not be a literal, word-for-word translation if it compromises naturalness.
Tone: Preserve the original tone of the input (e.g., formal, informal, neutral, sarcastic).
Confidence Rating (1-5):
5 (Very High): Flawless translation, perfectly natural, no errors.
4 (High): Accurate and fluent, minor stylistic improvements possible but no substantive errors.
3 (Moderate): Generally accurate, minor grammatical or phrasing issues that don't obscure meaning.
2 (Low): Several errors, fluency issues, meaning slightly ambiguous.
1 (Very Low): Significant errors, meaning is unclear or incorrect.

In [ ]:
!python3 src/translate_dataset.py

Loaded 50 existing predictions from temp_predictions.json.
Already processed 50 unique tasks.
Starting translation and prediction generation for sample tasks...
Processing tasks:   0%|                                | 0/1600 [00:00<?, ?it/s]

Processing tasks:   3%|▋                      | 51/1600 [00:14<07:16,  3.55it/s]

Processing tasks:   3%|▋                      | 52/1600 [00:22<12:37,  2.04it/s]

Processing tasks:   3%|▊                      | 53/1600 [00:29<19:25,  1.33it/s]

Processing tasks:   3%|▊                      | 54/1600 [00:36<27:42,  1.08s/it]

...

Processing tasks:  65%|█████████▊     | 1042/1600 [22:07:37<13:48:50, 89.12s/it]

Processing tasks:  65%|█████████▊     | 1043/1600 [22:08:57<13:22:58, 86.50s/it]

Processing tasks:  65%|█████████▊     | 1044/1600 [22:10:16<12:58:55, 84.06s/it]

Processing tasks:  65%|█████████▊     | 1045/1600 [22:11:41<13:01:04, 84.44s/it]

Processing tasks:  65%|█████████▊     | 1046/1600 [22:13:05<12:58:02, 84.26s/it]

In [ ]:
import requests

def send_push(title, message):
    url = "https://api.pushover.net/1/messages.json"
    payload = {
        "token": "akt9cug1jqigmhnswy2wnh9uvyyoa6",
        "user": "ufk3e4ok2wf77okk9vwhsw3nkiboa4",
        "title": title,
        "message": message
    }
    requests.post(url, data=payload)

# Example usage at end of notebook
send_push(
    "Notebook Completed ✅",
    "Your notebook finished successfully."
)